In [0]:
%run ../../config/utils

In [0]:
from datetime import datetime
import os
import sys
import pandas as pd
sys.path.append('..')
sys.path.append('../..')


from pyspark import SparkContext
from pyspark import SparkConf
from pyspark.sql import SparkSession
import pyspark.sql.functions as sqlf
from pyspark.sql.window import Window

from functools import reduce

from lib_cf.cf_io import (
    load_config,
    calculate_filepaths,
)

In [0]:
config_path = dbutils.widgets.get("config_path")
test = False if not dbutils.widgets.get("test") else True
rmse = False if not dbutils.widgets.get("rmse") else True
future = False if not dbutils.widgets.get("future") else True

In [0]:
# (1) ---- ARGUMENTS ---- #

CNF, CFG_PATH = load_config(config_path, test, future, rmse)
PARAMS = dict(list(CNF["shared"].items()) + list(CNF["combine"].items()))
PATHS = CNF["paths"]
OUTPUT_PATH = PATHS["MODEL"]
RUN_NAME = PARAMS["run_name"]
PARAMS, PATHS = calculate_filepaths(PARAMS, PATHS)

now = datetime.now().strftime("%Y%m%d")
columns = ["MBRSHP_SID", "CATEGORY_NAME", "CATEGORY_ID", "prediction"]
lambdas = [f"hs_ind_lambda{lmbd}" for lmbd in CNF["predict"]["lambda"]]

In [0]:
cf_data = read_cf_tables(cf_prediction, PARAMS, combined=True)

windowSpec = Window.partitionBy("MBRSHP_SID").orderBy("col2")
top_cat_list = []
for lambda_ in lambdas:
    windowSpec = Window.partitionBy("MBRSHP_SID", lambda_).orderBy(
        sqlf.desc("prediction")
    )
    top_cat = (
        cf_data.select(*columns, lambda_)
        .withColumn("rank", sqlf.row_number().over(windowSpec))
        .filter(sqlf.col("rank") == 1)
        .groupBy("CATEGORY_ID", "CATEGORY_NAME", lambda_)
        .agg(sqlf.countDistinct("MBRSHP_SID").alias("mbrs_count"))
        .withColumn(
            "cat_rank",
            sqlf.row_number().over(
                Window.partitionBy(lambda_).orderBy(sqlf.desc("mbrs_count"))
            ),
        )
        .filter(sqlf.col("cat_rank") <= 5)
        .withColumn("lambda", sqlf.lit(lambda_))
        .withColumnRenamed(lambda_, "hook_stretch")
    )
    top_cat_list.append(top_cat)


top_cat_df = reduce(lambda x, y: x.union(y), top_cat_list)

top_cat_df = top_cat_df.withColumn(
    "lambda",
    sqlf.substring(sqlf.col("lambda"), 14, sqlf.length(sqlf.col("lambda")))
)

top_cat_df.limit(5).display()

In [0]:
top_cat_df = top_cat_df.withColumn("RUN_NAME", f.to_date(f.lit(PARAMS["run_name"][-10:]), "yyyy_MM_dd"))

top_cat_df.write.mode('overwrite').option('replaceWhere', f"RUN_NAME = '{PARAMS['run_name'][-10:].replace('_','-')}'").saveAsTable(cf_top_cat)